# S&P 500 Options: The Risk-Overlay Boundary

In the other case studies this stage adds a risk overlay: a rule sitting on top of the
allocator's weights that caps a position, scales the book down after a drawdown, or targets a
volatility. The overlay is expressed as a target-weight transformation, which works because in
those case studies a position is a quantity of one instrument and its risk moves with that
quantity.

A short straddle does not have that shape here. Scaling the number of contracts would in fact
scale the legs, the hedge, the costs and the dollar Greeks together, so the objection is not that
option risk is independent of quantity. It is that the option engine never sees a quantity. It
holds five weekly cohorts at a fixed fifth of capital each and normalizes the weights inside a
cohort to sum to one, so an overlay that scales those weights down is renormalized straight back
up: there is no cash position for the book to move into. The execution path therefore refuses a
risk block rather than accept one it would silently discard. The controls that do govern this
strategy - the delta-hedge threshold, the settlement convention, the entry cost model, how many
weekly cohorts run at once - are fields of the strategy specification itself, fixed in
`12_backtest`; `15_costs` afterwards varies one of them to measure what the result depends on.

So this case study declares no risk-overlay variants, and this notebook is where that is
checked rather than assumed. It resolves the candidate set that came out of
`13_portfolio_management`, shows that the configured risk request set is empty, demonstrates that
a risk request would be refused if one were configured, and confirms it wrote nothing.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# S&P 500 選擇權：風險覆蓋策略的邊界

在其他案例研究中，此階段會加入風險覆蓋策略：疊加在配置權重之上的規則，用以限制部位、在回撤後縮小整體持倉，或設定目標波動率。覆蓋策略表示為目標權重的轉換；其他案例可這樣做，因為其中的部位是單一工具的持有數量，其風險也會隨數量變動。

此處的空頭跨式組合並非這種形式。調整合約口數確實會同時調整雙腿、避險、成本及以金額計算的希臘字母風險，因此問題不是選擇權風險與數量無關。問題在於選擇權引擎根本看不到數量。它同時持有五個每週批次，每批固定分配五分之一資本，並將批次內權重正規化至總和為一；因此，覆蓋策略若調低這些權重，權重會立刻被重新正規化調高，整體持倉也沒有可轉入的現金部位。所以執行路徑會拒絕風險區塊，不會接受後又默默捨棄。真正控制此策略的項目，包括 Delta 避險門檻、結算慣例、進場成本模型，以及同時運行的每週批次數，都是策略規格本身的欄位，固定於 `12_backtest`；之後 `15_costs` 會變動其中一項，衡量結果對其依賴程度。

因此，本案例研究不宣告任何風險覆蓋變體；本 Notebook 會驗證此事，而非直接假定。它取得 `13_portfolio_management` 產生的候選集合，顯示設定的風險請求集合為空，示範若設定風險請求將遭拒絕，並確認沒有寫入任何內容。
<!-- ML4T_ZH_END c0000-p000 -->

This is the third of the four backtest stages, and the last one that could add a run to the
candidate pool. It registers none, so the pool `15_costs` prices and `18_strategy_analysis`
reports is the one `13_portfolio_management` left. Costs runs after this notebook rather than
beside it so that the last stage to select is the last stage to run.

**Learning objectives**

- Recognise when a generic portfolio control cannot be applied to an instrument, and say what
  about the instrument makes it inapplicable.
- Read a stage that deliberately produces no results, and check that claim against the registry
  rather than against the notebook's own narration.

**Book reference**: Chapter 19

**Prerequisites**: the finalized candidate set published by
[`13_portfolio_management`](13_portfolio_management.ipynb), and through it
[`12_backtest`](12_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0000-p001 -->
這是四個回測階段中的第三階段，也是最後一個可能將執行結果加入候選池的階段。它不登錄任何結果，因此 `15_costs` 定價、`18_strategy_analysis` 報告的候選池，就是 `13_portfolio_management` 留下的集合。成本階段在本 Notebook 之後執行，而非與它並行，使最後執行的階段也是最後可能進行選取的階段。

**學習目標**

- 辨識通用投資組合控制無法套用於某工具的情況，並說明工具的哪些特性造成限制。
- 讀取刻意不產生結果的階段，並對照登錄系統驗證其宣稱，而非只相信 Notebook 的敘述。

**書籍參考**：第 19 章

**先備條件**：由 [`13_portfolio_management`](13_portfolio_management.ipynb) 發布的最終候選集合，以及其上游的 [`12_backtest`](12_backtest.ipynb)。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""Validate the empty S&P 500 options risk-overlay request boundary."""

import polars as pl

from case_studies.research import CandidateSet, Result
from case_studies.sp500_options.research_workflow import (
    open_study,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import (
    get_portfolio_risk_controls,
    get_position_risk_controls,
)

CASE_STUDY = "sp500_options"
STRATEGY_CANDIDATES = "sp500-options-strategy-candidates-v1"

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""

## The candidate set that passes through

Every member is required to be a complete backtest before the set is allowed to move on, so a
partial result cannot reach selection by being carried through a stage that does nothing.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 傳遞至下一階段的候選集合

集合中的每個成員都必須是完整回測，集合才能繼續往下游傳遞，因此部分完成的結果不能藉由經過一個不做任何事的階段而進入選取。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
if EXECUTION_TIER != "canonical":
    raise ValueError("risk-boundary validation requires the canonical candidate set")
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
candidates = CandidateSet.one(study, name=STRATEGY_CANDIDATES)
if candidates.member_kind != "backtest":
    raise TypeError("the finalized strategy candidate set must contain backtests")
backtests_before = study.backtests.table()
members = backtests_before.filter(pl.col("backtest_hash").is_in(candidates.members))
if members.height != len(candidates.members) or members.filter(~pl.col("complete")).height:
    raise RuntimeError("the finalized strategy candidate set is incomplete")

In [4]:
pl.DataFrame(
    {
        "candidate_set": [candidates.name],
        "candidate_set_hash": [candidates.hash],
        "member_count": [len(candidates.members)],
        "stages": [", ".join(sorted(members.get_column("stage").unique().to_list()))],
    }
)

candidate_set,candidate_set_hash,member_count,stages
str,str,i64,str
"""sp500-options-strategy-candida…","""37c528281e79""",846,"""allocation, signal"""


## The configured risk requests

Position-scope controls act on one holding, portfolio-scope controls act on the book. Both lists
come from `config/setup.yaml`, and both are empty for this case study. Reading them rather than
writing the emptiness into the notebook is what makes this a check: adding a control to the
configuration makes the cell below raise instead of quietly running an overlay the option path
cannot represent.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 設定中的風險請求

部位層級控制作用於單一持倉，投資組合層級控制作用於整體持倉。兩份清單都來自 `config/setup.yaml`，在本案例研究中均為空。讀取設定，而非直接在 Notebook 中寫死空清單，才讓這項作業成為檢查：若在設定中加入控制，下方儲存格會引發錯誤，而不會默默執行選擇權路徑無法表示的覆蓋策略。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
risk_rows = [
    {"scope": "position", **request} for request in get_position_risk_controls(CASE_STUDY)
] + [{"scope": "portfolio", **request} for request in get_portfolio_risk_controls(CASE_STUDY)]
risk_requests = (
    pl.DataFrame(risk_rows)
    if risk_rows
    else pl.DataFrame(schema={"scope": pl.String, "name": pl.String, "method": pl.String})
)
if not risk_requests.is_empty():
    raise RuntimeError("risk variants require an implemented typed options path before execution")
risk_requests

scope,name,method
str,str,str


## What happens to a risk request that is submitted anyway

The refusal lives in the execution path, not in this notebook, so it holds for a reader who
writes their own request as well. The cell below builds one against the highest-Sharpe candidate
and confirms it is rejected before anything is fitted or written.

The probe opens that candidate and copies its strategy verbatim, so the risk block is the only
thing about the request that is new. Substituting a signal of the notebook's own would make the
refusal a statement about that substitute rather than about a candidate the pipeline produced.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 若仍提交風險請求，會發生什麼事

拒絕機制位於執行路徑，而非本 Notebook，因此讀者自行撰寫請求時也同樣適用。下方儲存格針對夏普比率最高的候選結果建立請求，確認它在任何模型擬合或資料寫入之前遭到拒絕。

這項探查會開啟該候選結果並逐字複製其策略，因此請求中只有風險區塊是新增的。若改用 Notebook 自訂的訊號，拒絕結果只能說明替代訊號，而無法說明流程實際產生的候選結果。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
probe_member = members.sort("sharpe", "backtest_hash", descending=[True, False]).row(0, named=True)
probe_strategy = Result.open(study, probe_member["backtest_hash"]).spec()["strategy"]
probe = strategy_request_frame(
    [
        {
            "request_name": "risk-overlay-probe",
            "prediction_hash": probe_member["prediction_hash"],
            "label": probe_member["label"],
            "signal": probe_strategy["signal"],
            "allocation": probe_strategy.get("allocation"),
            "risk": {"name": "position_cap", "method": "max_weight", "max_weight": 0.1},
            "costs": probe_strategy.get("costs"),
            "chapter": "ch19",
        }
    ]
)
try:
    run_official_backtest_requests(study, probe, population_name=None)
except ValueError as refusal:
    # The refusal has to name the risk overlay. The request also carries the candidate's costs
    # block, which this path refuses separately, so accepting any ValueError would let a refusal
    # about costs be reported as the risk boundary holding.
    if "risk overlay" not in str(refusal):
        raise RuntimeError(
            f"the request was refused for something other than risk: {refusal}"
        ) from refusal
    print(f"risk request refused: {refusal}")
else:
    raise RuntimeError("the option execution path accepted a risk overlay it cannot represent")

risk request refused: the specialized option path does not support risk overlays


## Nothing was written

The registry is read back and compared against the snapshot taken before the probe. This is the
claim the stage makes, so it is checked against the store rather than against a counter this
notebook keeps.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 沒有寫入任何內容

重新讀取登錄系統，與探查前取得的快照比較。這正是此階段提出的宣稱，因此應對照儲存內容檢查，而非依賴 Notebook 自行維護的計數器。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
backtests_after = study.backtests.table()
if backtests_after.height != backtests_before.height:
    raise RuntimeError("the empty risk boundary wrote a backtest result")
if set(backtests_after.get_column("backtest_hash")) != set(
    backtests_before.get_column("backtest_hash")
):
    raise RuntimeError("the empty risk boundary changed the published backtest set")
pl.DataFrame(
    {
        "check": ["configured risk requests", "backtests before", "backtests after"],
        "value": [
            str(risk_requests.height),
            str(backtests_before.height),
            str(backtests_after.height),
        ],
    }
)

check,value
str,str
"""configured risk requests""","""0"""
"""backtests before""","""846"""
"""backtests after""","""846"""


## Key takeaways

- A portfolio control is defined against a representation of a position. When the engine holds a
  fully invested book of normalized cohort weights, there is no quantity for a target-weight
  overlay to act on, whatever the instrument.
- A stage that produces nothing still has to prove it, and the proof is the store's contents
  before and after, not a statement in the notebook.
- Refusing an unsupported request in the shared execution path, rather than in the notebook, is
  what makes the boundary hold for a reader's own requests too.

**Known limitations**: this says nothing about whether risk controls on a short-volatility book
are a good idea, only that the generic target-weight form cannot express them here. Implementing
them would mean an option-aware overlay acting on cohort membership, contract selection or the
hedge rule, and that is a change to the strategy specification rather than a stage on top of it.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 重點摘要

- 投資組合控制依某種部位表示方式定義。當引擎持有由正規化批次權重構成、資本全數投入的整體持倉時，無論交易工具為何，都沒有可供目標權重覆蓋策略操作的數量。
- 即使某個階段不產生任何內容，仍須加以證明；證據是儲存系統前後的內容，而非 Notebook 中的一句話。
- 在共用執行路徑拒絕不支援的請求，而非只在 Notebook 中拒絕，才能讓此邊界同樣約束讀者自行提出的請求。

**已知限制**：這並未判斷空頭波動率持倉是否適合風險控制，只說明通用目標權重形式在此無法表達這些控制。若要實作，必須建立理解選擇權的覆蓋策略，作用於批次成員、合約選取或避險規則；這屬於策略規格的修改，而非疊加在其上的階段。
<!-- ML4T_ZH_END c0012-p000 -->
